In [ ]:
from src.paths import DATA_ROOT
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import numpy as np
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from pathlib import Path

In [ ]:


# Configuration
MAX_SAMPLES_PER_DATASET = 500
BATCH_SIZE = 32
SEED = 42

# Your 5 datasets
datasets_config = {
    'Tufts': f'{DATA_ROOT}/raw_tufts',
    'USPFORP': f'{DATA_ROOT}/raw_uspforp',
    'ADLD C1': f'{DATA_ROOT}/raw_sorted_dual_labeled/dl-client1',
    'ADLD C2': f'{DATA_ROOT}/raw_sorted_dual_labeled/dl-client2',
    'DENTEX 2': f'{DATA_ROOT}/raw_sorted_dentex/client1',
    'DENTEX 3': f'{DATA_ROOT}/raw_sorted_dentex/client2',
    'DENTEX 1': f'{DATA_ROOT}/raw_sorted_dentex/client0',
}

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}

def list_images(folder: Path) -> list:
    return sorted([p for p in folder.iterdir() if p.suffix.lower() in IMAGE_EXTS])

print("="*70)
print("t-SNE Visualization using ResNet50 Features")
print("5 Dental Image Datasets (Grayscale)")
print("="*70)

# Load pretrained ResNet50
print("\nLoading ResNet50...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

resnet50 = models.resnet50(pretrained=True)
resnet50 = torch.nn.Sequential(*list(resnet50.children())[:-1])
resnet50.eval()
resnet50 = resnet50.to(device)
print("✓ ResNet50 loaded (2048-dim features)")

# Image preprocessing for grayscale images
class GrayscaleToRGB:
    """Convert grayscale to RGB by repeating channels"""
    def __call__(self, img):
        if img.mode == 'L':  # Grayscale
            return img.convert('RGB')
        elif img.mode == 'I' or img.mode == 'I;16':  # 16-bit grayscale
            # Normalize to 8-bit range and convert to RGB
            img_array = np.array(img)
            img_array = ((img_array - img_array.min()) / (img_array.max() - img_array.min()) * 255).astype(np.uint8)
            img_8bit = Image.fromarray(img_array, mode='L')
            return img_8bit.convert('RGB')
        else:
            return img.convert('RGB')

transform = transforms.Compose([
    GrayscaleToRGB(),  # Convert grayscale to RGB first
    transforms.Resize((224, 224)),
    #transforms.CenterCrop(224),
    transforms.ToTensor(),
    #transforms.Normalize(mean=[0.485, 0.456, 0.406], 
    #                    std=[0.229, 0.224, 0.225])
])

#transform = transforms.Compose([
#    GrayscaleToRGB(),  # Convert grayscale to RGB first
#    transforms.Resize(256),
#    transforms.CenterCrop(224),
#    transforms.ToTensor(),
#    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
#                        std=[0.229, 0.224, 0.225])
#])

# Custom dataset
class DentalImageDataset(Dataset):
    def __init__(self, image_paths, dataset_ids, transform=None):
        self.image_paths = image_paths
        self.dataset_ids = dataset_ids
        self.transform = transform
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        try:
            img = Image.open(img_path)
            
            if self.transform:
                img = self.transform(img)
            
            return img, self.dataset_ids[idx], img_path
        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            # Return a blank tensor in case of error
            return torch.zeros(3, 224, 224), self.dataset_ids[idx], img_path

# Collect and sample images
np.random.seed(SEED)

sampled_image_paths = []
sampled_dataset_ids = []

for dataset_idx, (dataset_name, dataset_path) in enumerate(datasets_config.items()):
    print(f"\nProcessing {dataset_name}...")
    print(f"  Path: {dataset_path}")
    
    image_list = list_images(Path(dataset_path))
    
    if len(image_list) == 0:
        print(f"  ⚠️ No images found!")
        continue
    
    # Check image bit depth
    sample_img = Image.open(image_list[0])
    print(f"  Sample image mode: {sample_img.mode}, size: {sample_img.size}")
    
    if len(image_list) > MAX_SAMPLES_PER_DATASET:
        sampled_imgs = np.random.choice(image_list, MAX_SAMPLES_PER_DATASET, replace=False).tolist()
        print(f"  Sampled {MAX_SAMPLES_PER_DATASET}/{len(image_list)} images")
    else:
        sampled_imgs = image_list
        print(f"  Using all {len(image_list)} images")
    
    sampled_image_paths.extend([str(p) for p in sampled_imgs])
    sampled_dataset_ids.extend([dataset_idx] * len(sampled_imgs))

print(f"\n{'='*70}")
print(f"Total samples collected: {len(sampled_image_paths)}")

# Create dataset and dataloader
dataset = DentalImageDataset(
    sampled_image_paths, 
    sampled_dataset_ids, 
    transform=transform
)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, 
                       num_workers=0, pin_memory=True if device.type == 'cuda' else False)

# Extract features
print("\nExtracting ResNet50 features...")
all_features = []
all_dataset_ids = []

with torch.no_grad():
    for batch_idx, (images, dataset_ids, paths) in enumerate(dataloader):
        images = images.to(device)
        
        features = resnet50(images)
        features = features.squeeze(-1).squeeze(-1)
        
        all_features.append(features.cpu().numpy())
        all_dataset_ids.extend(dataset_ids.numpy())
        
        if (batch_idx + 1) % 10 == 0:
            print(f"  Processed {(batch_idx + 1) * BATCH_SIZE}/{len(dataset)} images")

all_features = np.vstack(all_features)
all_dataset_ids = np.array(all_dataset_ids)

print(f"\n{'='*70}")
print(f"Feature extraction complete!")
print(f"Feature shape: {all_features.shape}")

# Print distribution
print("\nSample distribution:")
dataset_names = list(datasets_config.keys())
for dataset_idx, dataset_name in enumerate(dataset_names):
    mask = all_dataset_ids == dataset_idx
    print(f"  {dataset_name}: {mask.sum()} images")

# PCA
print(f"\n{'='*70}")
print("Applying PCA...")
pca = PCA(n_components=50, random_state=SEED)
features_pca = pca.fit_transform(all_features)
print(f"PCA explained variance: {pca.explained_variance_ratio_.sum():.3f}")

print("\nApplying t-SNE (this may take a few minutes)...")
tsne = TSNE(
    n_components=2, 
    perplexity=30,
    random_state=SEED, 
    max_iter=1000,  # Changed from n_iter to max_iter
    verbose=1
)
features_tsne = tsne.fit_transform(features_pca)
print("✓ t-SNE complete!")

# Visualization 1: Main plot
print("\nGenerating visualizations...")

fig, ax = plt.subplots(figsize=(10, 10))

colors =  colors = ['blue', 'orange', 'cyan', 'red', 'purple', 'green', 'pink']

for dataset_idx, dataset_name in enumerate(dataset_names):
    mask = all_dataset_ids == dataset_idx
    n_samples = mask.sum()
    
    if n_samples == 0:
        continue
    
    alpha = min(0.8, 500 / n_samples)
    size = max(30, 300 / np.sqrt(n_samples))
    
    ax.scatter(
        features_tsne[mask, 0],
        features_tsne[mask, 1],
        c=colors[dataset_idx],
        label=f'{dataset_name} (n={n_samples})',
        alpha=alpha,
        s=size,
        edgecolors='black',
        linewidths=0.5
    )

#ax.set_title('t-SNE: 5 Dental Datasets (Grayscale, ResNet50 Features)', 
#             fontsize=16, fontweight='bold', pad=20)
#ax.set_xlabel('t-SNE Component 1', fontsize=12)
#ax.set_ylabel('t-SNE Component 2', fontsize=12)
ax.legend(loc='best', framealpha=0.95, fontsize=15)
ax.tick_params(axis='both', which='major', labelsize=15)
#ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('tsne_5_datasets_grayscale.png', dpi=300, bbox_inches='tight')
print("✓ Saved: tsne_5_datasets_grayscale.png")
plt.show()

# Visualization 2: Individual dataset distributions in 2x4 grid
fig, axes = plt.subplots(2, 4, figsize=(24, 12))
axes = axes.flatten()

for dataset_idx, dataset_name in enumerate(dataset_names):
    ax = axes[dataset_idx]
    dataset_mask = all_dataset_ids == dataset_idx
    
    ax.scatter(
        features_tsne[~dataset_mask, 0],
        features_tsne[~dataset_mask, 1],
        c='lightgray',
        alpha=0.2,
        s=10,
        label='Other datasets'
    )
    
    if dataset_mask.sum() > 0:
        ax.scatter(
            features_tsne[dataset_mask, 0],
            features_tsne[dataset_mask, 1],
            c=colors[dataset_idx],
            label=f'{dataset_name}',
            alpha=0.7,
            s=60,
            edgecolors='black',
            linewidths=0.5
        )
    
    ax.set_title(f'{dataset_name} (n={dataset_mask.sum()})', 
                fontsize=12, fontweight='bold')
    ax.set_xlabel('t-SNE Component 1')
    ax.set_ylabel('t-SNE Component 2')
    ax.legend(loc='best', fontsize=9, framealpha=0.9)
    ax.grid(True, alpha=0.3)

# Hide unused subplot (index 7)
axes[7].axis('off')

plt.suptitle('t-SNE: Individual Dataset Distributions', 
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('tsne_5_datasets_individual_grayscale.png', dpi=300, bbox_inches='tight')
print("✓ Saved: tsne_5_datasets_individual_grayscale.png")
plt.show()

print(f"\n{'='*70}")
print("Visualization complete!")
print("="*70)

In [ ]:
# Visualization 1: Main plot
print("\nGenerating visualizations...")

fig, ax = plt.subplots(figsize=(14, 10))

colors =  colors = ['blue', 'orange', 'cyan', 'red', 'purple', 'green', 'pink']

for dataset_idx, dataset_name in enumerate(dataset_names):
    mask = all_dataset_ids == dataset_idx
    n_samples = mask.sum()
    
    if n_samples == 0:
        continue
    
    alpha = min(0.8, 500 / n_samples)
    size = max(40, 300 / np.sqrt(n_samples))
    
    ax.scatter(
        features_tsne[mask, 0],
        features_tsne[mask, 1],
        c=colors[dataset_idx],
        label=f'{dataset_name} (n={n_samples})',
        alpha=alpha,
        s=size,
        edgecolors='black',
        linewidths=0.5
    )

#ax.set_title('t-SNE: 5 Dental Datasets (Grayscale, ResNet50 Features)', 
#             fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('t-SNE Component 1', fontsize=26)
ax.set_ylabel('t-SNE Component 2', fontsize=26)
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), framealpha=0.95, fontsize=26)
ax.tick_params(axis='both', which='major', labelsize=26)
#ax.grid(True, alpha=0.3, linestyle='--')

plt.tight_layout()
plt.savefig('tsne_5_datasets_grayscale.png', dpi=300, bbox_inches='tight')
print("✓ Saved: tsne_5_datasets_grayscale.png")
plt.show()

# Visualization 2: Individual dataset distributions in 2x4 grid
fig, axes = plt.subplots(2, 4, figsize=(24, 12))
axes = axes.flatten()

for dataset_idx, dataset_name in enumerate(dataset_names):
    ax = axes[dataset_idx]
    dataset_mask = all_dataset_ids == dataset_idx
    
    ax.scatter(
        features_tsne[~dataset_mask, 0],
        features_tsne[~dataset_mask, 1],
        c='lightgray',
        alpha=0.2,
        s=10,
        label='Other datasets'
    )
    
    if dataset_mask.sum() > 0:
        ax.scatter(
            features_tsne[dataset_mask, 0],
            features_tsne[dataset_mask, 1],
            c=colors[dataset_idx],
            label=f'{dataset_name}',
            alpha=0.7,
            s=60,
            edgecolors='black',
            linewidths=0.5
        )
    
    ax.set_title(f'{dataset_name} (n={dataset_mask.sum()})', 
                fontsize=12, fontweight='bold')
    ax.set_xlabel('t-SNE Component 1')
    ax.set_ylabel('t-SNE Component 2')
    ax.legend(loc='best', fontsize=9, framealpha=0.9)
    ax.grid(True, alpha=0.3)

# Hide unused subplot (index 7)
axes[7].axis('off')

plt.suptitle('t-SNE: Individual Dataset Distributions', 
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('tsne_5_datasets_individual_grayscale.png', dpi=300, bbox_inches='tight')
print("✓ Saved: tsne_5_datasets_individual_grayscale.png")
plt.show()

print(f"\n{'='*70}")
print("Visualization complete!")
print("="*70)